# L07-03｜Qwen3-0.6B LoRA 训练与 loss 分析


## 先写下判断

完整训练用来一起观察参数、训练过程和 loss 曲线。运行前写下一个猜测：loss 整体下降、来回波动或突然异常时，可能发生了什么？


## 初始化：每个 Notebook 都从零开始

下面的代码安装缺失依赖、检查 NPU、从 ModelScope 下载 `Qwen/Qwen3-0.6B`，并生成本次训练使用的 JSONL 数据。ModelArts 镜像需要使用 Python 3.10–3.12，并带有匹配的 Ascend/CANN/`torch_npu` 运行时。本 Notebook 不读取 L07-02 的输出，单独放到新的 ModelArts 实例也能从第一格运行到最后一格。


In [ ]:
from __future__ import annotations

import importlib.util
import json
import math
import os
import shlex
import shutil
import site
import subprocess
import sys
from pathlib import Path

def require(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)

require((3, 10) <= sys.version_info[:2] <= (3, 12), '本流程需要 Python 3.10、3.11 或 3.12；请使用带匹配 Ascend 运行时的 ModelArts 镜像。')
def install_missing_packages() -> None:
    packages = {'modelscope': 'modelscope', 'swift': 'ms-swift', 'matplotlib': 'matplotlib'}
    missing = [dist for module, dist in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        print('安装缺失依赖：', missing)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', *missing])
    python_bin = str(Path(sys.executable).parent)
    user_bin = str(Path(site.getuserbase()) / 'bin')
    old_path = os.environ.get('PATH', '')
    path_entries = old_path.split(os.pathsep) if old_path else []
    for candidate in (python_bin, user_bin):
        if candidate not in path_entries:
            old_path = candidate + os.pathsep + old_path
            path_entries.insert(0, candidate)
    os.environ['PATH'] = old_path

install_missing_packages()

def load_ascend_env() -> None:
    candidates = [
        Path('/usr/local/Ascend/ascend-toolkit/set_env.sh'),
        Path('/usr/local/Ascend/ascend-toolkit/latest/set_env.sh'),
    ]
    ascend_root = Path('/usr/local/Ascend')
    if ascend_root.is_dir():
        candidates.extend(sorted(ascend_root.glob('**/set_env.sh')))
    seen = set()
    for script in candidates:
        if not script.is_file() or str(script) in seen:
            continue
        seen.add(str(script))
        result = subprocess.run(
            ['bash', '-lc', f'source {shlex.quote(str(script))} >/dev/null 2>&1 && env -0'],
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=False,
        )
        if result.returncode != 0:
            continue
        for item in result.stdout.split(b'\0'):
            if b'=' in item:
                key, value = item.split(b'=', 1)
                os.environ[key.decode()] = value.decode(errors='ignore')
        print('已加载 Ascend 环境：', script)
        return
    print('未找到 CANN set_env.sh；继续使用当前 ModelArts 进程环境。')

load_ascend_env()
os.environ.setdefault('ASCEND_RT_VISIBLE_DEVICES', '0')
import torch
try:
    import torch_npu  # noqa: F401
except Exception as exc:
    raise RuntimeError('当前环境无法导入 torch_npu。请使用带 Ascend/CANN 运行时的 ModelArts 镜像。') from exc
require(hasattr(torch, 'npu'), '当前 PyTorch 没有 torch.npu；请检查 ModelArts 的 Ascend 运行时。')
npu_count = torch.npu.device_count()
require(npu_count > 0, '没有检测到 NPU。请确认 ModelArts 实例规格和可见设备。')
torch_version = getattr(torch, '__version__', 'unknown')
torch_npu_version = getattr(torch_npu, '__version__', 'unknown')
def major_minor(version: str) -> tuple[int, int] | None:
    try:
        parts = version.split('+', 1)[0].split('.')
        return int(parts[0]), int(parts[1])
    except (IndexError, ValueError):
        return None
if major_minor(torch_version) and major_minor(torch_npu_version):
    require(major_minor(torch_version) == major_minor(torch_npu_version), f'torch 与 torch_npu 版本不匹配：{torch_version} vs {torch_npu_version}。请按同一套 CANN/PyTorch/torch_npu 重新准备 ModelArts 镜像。')
torch.npu.set_device(0)
probe = torch.zeros(1, device='npu:0')
del probe

NOTEBOOK_ID = 'L07-03'
WORK_DIR = Path(os.environ.get('L07_WORK_DIR', str(Path.cwd() / f'{NOTEBOOK_ID}_workspace'))).expanduser()
WORK_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = os.environ.get('L07_MODEL_ID', 'Qwen/Qwen3-0.6B')
MODEL_CACHE = Path(os.environ.get('MODELSCOPE_CACHE', str(WORK_DIR / 'modelscope_cache'))).expanduser()
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
MODEL_PATH_OVERRIDE = os.environ.get('L07_MODEL_PATH')
if MODEL_PATH_OVERRIDE:
    MODEL_PATH = Path(MODEL_PATH_OVERRIDE).expanduser()
    require(MODEL_PATH.is_dir(), f'指定的模型目录不存在：{MODEL_PATH}')
else:
    from modelscope import snapshot_download
    MODEL_PATH = Path(snapshot_download(MODEL_ID, cache_dir=str(MODEL_CACHE)))
require((MODEL_PATH / 'config.json').is_file(), f'模型目录缺少 config.json：{MODEL_PATH}')
MODEL_REF = str(MODEL_PATH)

def qwen3_answer(text: str) -> str:
    return '<think>\n\n</think>\n\n' + text

examples = [
    ('请用一句话解释 LoRA 与全参数微调的区别。', 'LoRA 只训练注入的低秩适配器参数，全参数微调会更新模型的全部参数。'),
    ('Python 中如何获取列表长度？', '使用内置函数 len，例如 len([1, 2, 3]) 的结果是 3。'),
    ('一个 batch 有 2 条样本，梯度累积 4 步，完成一次更新前处理多少条样本？', '在没有丢弃样本的情况下，会先处理 2 乘以 4，也就是 8 条样本。'),
    ('把“先检查日志，再判断原因”翻译成英文。', 'Check the logs first, then identify the cause.'),
    ('为什么训练 loss 下降不能单独证明模型效果更好？', '因为 loss 只反映训练目标，还需要独立评估集和明确的评价指标。'),
    ('请说出一次可追溯训练至少要保留的一项证据。', '可以保留原始 logging.jsonl、启动命令、配置或 checkpoint 路径。'),
]
records = [
    {'messages': [
        {'role': 'system', 'content': '你是一个简洁、准确的课程实验助手。'},
        {'role': 'user', 'content': question + ' /no_think'},
        {'role': 'assistant', 'content': qwen3_answer(answer)},
    ]}
    for question, answer in examples
]
TRAIN_DATA = WORK_DIR / 'train.jsonl'
with TRAIN_DATA.open('w', encoding='utf-8') as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
loaded_records = [json.loads(line) for line in TRAIN_DATA.read_text(encoding='utf-8').splitlines() if line.strip()]
require(len(loaded_records) == len(records), '生成的 JSONL 条数不一致。')
require(all('messages' in item for item in loaded_records), '每条训练数据都必须包含 messages。')
environment_record = {'python': sys.version.split()[0], 'torch': getattr(torch, '__version__', 'unknown'), 'torch_npu': getattr(torch_npu, '__version__', 'unknown'), 'npu_count': npu_count, 'visible_npus': os.environ['ASCEND_RT_VISIBLE_DEVICES'], 'model_id': MODEL_ID, 'model_path': str(MODEL_PATH), 'train_data': str(TRAIN_DATA)}
(WORK_DIR / 'environment.json').write_text(json.dumps(environment_record, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

SWIFT_BIN = shutil.which('swift')
require(SWIFT_BIN is not None, '找不到 swift 命令；请确认 ms-swift 已安装且当前 Python 的 bin 目录在 PATH 中。')
VISIBLE_NPUS = os.environ['ASCEND_RT_VISIBLE_DEVICES']
print({'model_id': MODEL_ID, 'model_path': MODEL_REF, 'train_data': str(TRAIN_DATA), 'work_dir': str(WORK_DIR), 'npu_count': npu_count, 'visible_npus': VISIBLE_NPUS})

def run_streaming(command: list[str], log_path: Path, env: dict[str, str]) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('将执行：\n' + shlex.join(command))
    with log_path.open('w', encoding='utf-8') as stream:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            stream.write(line)
        returncode = process.wait()
    require(returncode == 0, f'命令失败（退出码 {returncode}）。请检查：{log_path}')
    print('命令输出已保存到：', log_path)


In [ ]:
# 这些参数是本节观察 LoRA 训练过程的起点。
LORA_RANK = 8
LORA_ALPHA = 32
LEARNING_RATE = 1e-4
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
MAX_LENGTH = 512
print('环境、模型和本地 JSONL 数据已准备好。')


In [ ]:
def read_first_record(path: Path) -> dict:
    raw = path.read_text(encoding='utf-8').lstrip()
    require(raw, f'数据文件为空：{path}')
    if raw.startswith('['):
        records = json.loads(raw)
        require(isinstance(records, list) and records and isinstance(records[0], dict), 'JSON 数组的第一项必须是对象。')
        return records[0]
    return json.loads(next(line for line in raw.splitlines() if line.strip()))

first_record = read_first_record(TRAIN_DATA)
require(isinstance(first_record, dict), '训练数据的首条记录必须是对象。')
keys = sorted(first_record.keys())
accepted = {'messages', 'conversations', 'instruction', 'query', 'response'}
require(set(keys) & accepted, f'未识别到常用指令数据字段，当前仅看到：{keys}')
if 'messages' in first_record:
    require(isinstance(first_record['messages'], list) and first_record['messages'], 'messages 必须是非空列表。')
    require({'role', 'content'} <= set(first_record['messages'][0]), 'messages 的首项至少需要 role 与 content 字段。')
print('数据结构检查通过；仅显示字段名，不展示样本内容：', keys)


## 训练设计：假设和参数分开改

如果要比较不同设置，一次只改一个因素，或只改一组有明确关系的因素。启动训练前，写清楚下面几项：

| 项目 | 训练前要写清楚的问题 |
| --- | --- |
| 目标 | 这次运行要观察什么？例如“确认一个完整 epoch 能完成” |
| 保持不变 | 模型、数据版本、最大序列长度、随机种子、资源条件等 |
| 计划改变 | 例如 `lora_rank` 或学习率；一次别改所有变量 |
| 观察证据 | `logging.jsonl`、命令输出、checkpoint、异常日志 |
| 停止条件 | 遇到 OOM、NaN、数据错误或中断时，在哪里记录并停止 |


## 训练配置

本 Notebook 训练 1 个 epoch。一个 epoch 就是完整遍历一遍当前 JSONL 数据。


### 这几个设置为什么这样写

- `NUM_TRAIN_EPOCHS=1` 只是教学起点，不代表通用最优值。本次运行会完整遍历这份数据。
- `SAVE_STEPS` 控制 checkpoint 间隔。太小会频繁写盘，太大则可能没有可恢复的中间状态。
- `LOGGING_STEPS` 控制 loss 的记录频率。这里设为 1，是为了让这份小数据尽量留下完整轨迹。
- `NPROC_PER_NODE` 是训练进程数。默认值为 1；只有确认多卡环境和启动方式后，才应改成更大的数。

每次少改几个变量，曲线变化才更容易和具体原因对应起来。


In [ ]:
# 用完整训练观察 loss 随训练推进的变化。
RUN_NAME = 'L07-03_train'
RUN_DIR = WORK_DIR / RUN_NAME
NUM_TRAIN_EPOCHS = 1
SAVE_STEPS = 4
LOGGING_STEPS = 1
NPROC_PER_NODE = int(os.environ.get('L07_NPROC_PER_NODE', '1'))

train_command = [
    SWIFT_BIN, 'sft',
    '--model', MODEL_REF,
    '--dataset', str(TRAIN_DATA),
    '--torch_dtype', 'bfloat16',
    '--tuner_type', 'lora',
    '--target_modules', 'all-linear',
    '--lora_rank', str(LORA_RANK),
    '--lora_alpha', str(LORA_ALPHA),
    '--loss_scale', 'ignore_empty_think',
    '--num_train_epochs', str(NUM_TRAIN_EPOCHS),
    '--per_device_train_batch_size', str(PER_DEVICE_BATCH_SIZE),
    '--gradient_accumulation_steps', str(GRADIENT_ACCUMULATION_STEPS),
    '--learning_rate', str(LEARNING_RATE),
    '--max_length', str(MAX_LENGTH),
    '--logging_steps', str(LOGGING_STEPS),
    '--save_steps', str(SAVE_STEPS),
    '--save_total_limit', '2',
    '--output_dir', str(RUN_DIR),
]
train_env = os.environ.copy()
train_env['ASCEND_RT_VISIBLE_DEVICES'] = VISIBLE_NPUS
train_env['NPROC_PER_NODE'] = str(NPROC_PER_NODE)
train_env['PYTHONUNBUFFERED'] = '1'
print('NPROC_PER_NODE:', NPROC_PER_NODE)
print('训练配置已生成。')


### 训练参数和图上的点

`LOGGING_STEPS` 决定隔多少 step 记录一次 loss，`SAVE_STEPS` 决定隔多少 step 保存一次 checkpoint。图上的点有多密，主要取决于日志间隔。点数不等于训练步数。


## 运行训练

运行时看 step 是否持续增加，loss 是否按设定间隔出现。完整输出会写入 notebook_stdout.log。


In [ ]:
stdout_path = RUN_DIR / 'notebook_stdout.log'
run_streaming(train_command, stdout_path, train_env)


### 训练时重点看什么

看日志是否持续产生、step 是否推进、进程是否异常。单个 loss 点通常解释不了训练状态，不必盯着一个点反复猜。

如果遇到 OOM、NaN、进程退出或长时间不动，先保留 `notebook_stdout.log`。记录发生时的 step、报错原文、当时配置和排查动作，不要只写“训练失败”。


## 从原始日志画 loss

代码会从当前运行生成的 `logging.jsonl` 中读取 `loss` / `train_loss` 和 step。找不到这些字段时，应该先检查真实日志格式，不要手工补点。


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

logging_files = sorted(RUN_DIR.rglob('logging.jsonl'), key=lambda path: path.stat().st_mtime)
require(logging_files, f'没有找到 logging.jsonl。请检查 {stdout_path}')
logging_file = logging_files[-1]
run_dir = logging_file.parent
records = []
for raw in logging_file.read_text(encoding='utf-8').splitlines():
    try:
        record = json.loads(raw)
    except json.JSONDecodeError:
        continue
    loss = record.get('loss', record.get('train_loss'))
    if isinstance(loss, (int, float)) and math.isfinite(float(loss)):
        step = record.get('global_step', record.get('current_steps'))
        if not isinstance(step, (int, float)):
            for key in ('global_step/max_steps', 'iteration'):
                value = record.get(key)
                if isinstance(value, str) and '/' in value:
                    left = value.split('/', 1)[0].strip()
                    if left.isdigit():
                        step = int(left)
                        break
        records.append({'step': step if isinstance(step, (int, float)) else len(records) + 1, 'loss': float(loss), 'raw': record})
require(records, f'未从 {logging_file} 解析到 loss / train_loss 字段。请记录实际字段名，不要手填曲线。')
checkpoints = [path for path in run_dir.glob('checkpoint-*') if path.is_dir()]
require(checkpoints, f'没有在 {run_dir} 找到 checkpoint。')

steps = [item['step'] for item in records]
losses = [item['loss'] for item in records]
plt.figure(figsize=(8, 4.5))
plt.plot(steps, losses, marker='o', linewidth=1.2, markersize=3, label='raw loss')
if len(losses) >= 5:
    window = min(5, len(losses))
    smooth = [sum(losses[max(0, i-window+1):i+1]) / len(losses[max(0, i-window+1):i+1]) for i in range(len(losses))]
    plt.plot(steps, smooth, linewidth=2, label=f'moving average ({window})')
plt.xlabel('training step')
plt.ylabel('loss')
plt.title('L07-03 training loss (this run only)')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
figure_path = RUN_DIR / 'loss_curve.png'
plt.savefig(figure_path, dpi=150)
plt.show()
print('loss points:', len(records))
print('raw log:', logging_file)
print('checkpoint:', checkpoints[-1])
print('figure:', figure_path)


## 按证据读 loss 曲线

写分析时把事实和推测分开，至少交代这些内容：

1. 来源：横轴用的是什么 step 字段？纵轴是 `loss` 还是 `train_loss`？图来自哪一个 `logging.jsonl`？
2. 现象：整体趋势如何，波动范围多大，有没有突变或缺失点？
3. 上下文：结合日志间隔、学习率、batch、序列长度、数据顺序和异常事件，提出可能解释。
4. 边界：训练 loss 不能单独证明评估集效果，也不能独自说明模型优劣。

移动平均线只用来帮助看趋势。记录时写明窗口大小，还要保留原始点，不要用平滑线遮住异常。


## 可选：检查日志里的内存字段

只有本次 `logging.jsonl` 真的写出了 `memory(GiB)`，代码才会画内存曲线。它不等价于统一采样口径下的 NPU 峰值显存。


In [ ]:
memory_records = []
for item in records:
    value = item['raw'].get('memory(GiB)')
    if isinstance(value, (int, float)):
        memory_records.append((item['step'], float(value)))
if memory_records:
    x, y = zip(*memory_records)
    plt.figure(figsize=(8, 3.5))
    plt.plot(x, y, marker='o', linewidth=1.2)
    plt.xlabel('training step')
    plt.ylabel('memory (GiB)')
    plt.title('Memory records emitted by this run')
    plt.grid(alpha=0.25)
    plt.tight_layout()
    memory_path = RUN_DIR / 'memory_from_logging_jsonl.png'
    plt.savefig(memory_path, dpi=150)
    plt.show()
    print('memory figure:', memory_path)
else:
    print('此运行的 logging.jsonl 没有 memory(GiB) 字段；请在学习笔记中如实记下“未采集”，不要由 loss 或设备规格估算显存。')


## 资源记录能说明什么

只有 `logging.jsonl` 含有 `memory(GiB)` 时，下面才会照原值画图。这个字段表示训练框架在该日志口径下记录的内存，不自动等于“峰值显存”。

要比较两次资源占用，至少要保持或记录设备、batch size、最大序列长度、精度、梯度累积、采样时刻和并发任务。条件不同，两个数字就不能直接放在一起比。


## 训练后回顾

1. 这条曲线的横轴、纵轴、记录间隔和是否平滑分别是什么？
2. 写出两条可以直接从图上看到的事实，并标出仍然只是推测的部分。
3. 如果出现异常波动、NaN、OOM 或中断，记下发生位置、相关日志和排查步骤；没有就写“无”。
4. 这条曲线不能单独证明什么？
5. 下一次运行前，你还需要复查或改进什么？


## 回顾清单

- [ ] 我能说清本次模型、数据、配置、命令和环境。
- [ ] `logging.jsonl`、`notebook_stdout.log`、图表和 checkpoint / adapter 路径都能找到。
- [ ] loss 图来自原始日志，并写明了横轴、纵轴、记录间隔和是否平滑。
- [ ] 我把直接观察、可能解释和未验证事项分开写了。
- [ ] 如果发生 OOM、NaN、中断或重跑，学习笔记里保留了真实记录。
- [ ] 没有把未采集的显存、吞吐或评估结果补成数字。
